<a href="https://colab.research.google.com/github/jojojojo-jo/LRP_analysis_resnet/blob/main/Phase4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
from skimage.metrics import structural_similarity as ssim
from skimage.transform import resize
from scipy.stats import pearsonr, spearmanr
from scipy.ndimage import center_of_mass
from tqdm.notebook import tqdm

In [ ]:
baseline_csv = Path('/content/drive/MyDrive/study/Project/LRP_Heatmap/clean_heatmap/clean_lrp_metadata.csv')
perturbed_csv = Path('/content/drive/MyDrive/study/Project/LRP_Heatmap/perturbed_heatmap_new/perturbed_lrp_metadata.csv')

CLEAN_TENSORS_BASE_DIR = Path('/content/drive/MyDrive/study/Project/LRP_Heatmap/clean_tensors')
PERTURBED_TENSORS_BASE_DIR = Path('/content/drive/MyDrive/study/Project/LRP_Heatmap/perturbed_tensors')

output_dir = Path('/content/drive/MyDrive/study/Project/LRP_Heatmap/phase4_output')
results_csv = output_dir / 'phase4_results.csv'
checkpoint_csv = output_dir / 'phase4_results_checkpoint.csv'
output_dir.mkdir(parents=True, exist_ok=True)

CHECKPOINT_EVERY = 200  # save partial results every N rows


In [ ]:
CONF_RE = re.compile(r'([\d.]+)')
LEVEL_RE = re.compile(r'lvl_(\d+)_(.+)')
EPS_RE = re.compile(r'eps0*(\d+)')
DB_RE = re.compile(r'(\d+)\s*dB', re.IGNORECASE)
EPS_SCALE = 1000.0


def parse_intensity_level(lvl_name: str) -> dict:
    level_match = LEVEL_RE.match(lvl_name)
    level_number = int(level_match.group(1)) if level_match else None
    remainder = level_match.group(2) if level_match else lvl_name

    eps_match = EPS_RE.search(remainder)
    db_match = DB_RE.search(remainder)

    if eps_match is not None:
        intensity_value = float(eps_match.group(1)) / EPS_SCALE
        intensity_unit = 'epsilon'
    elif db_match is not None:
        intensity_value = float(db_match.group(1))
        intensity_unit = 'dB'
    else:
        intensity_value = np.nan
        intensity_unit = 'unknown'

    return {'level_number': level_number, 'intensity_value': intensity_value, 'intensity_unit': intensity_unit}


def parse_tensor_confidence(value) -> float:
    """Extracts a float out of strings like \"tensor(0.9372, device='cuda:0')\"."""
    if isinstance(value, (int, float)):
        return float(value)
    match = CONF_RE.search(str(value))
    return float(match.group(1)) if match else np.nan


In [ ]:
def normalize_map(arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float64)
    a_min, a_max = arr.min(), arr.max()
    if (a_max - a_min) < 1e-12:
        return np.zeros_like(arr)
    return (arr - a_min) / (a_max - a_min)


def align_shapes(orig: np.ndarray, pert: np.ndarray):
    if orig.shape != pert.shape:
        pert = resize(pert, orig.shape, mode='reflect', anti_aliasing=True)
    return orig, pert


def compute_ssim(orig: np.ndarray, pert: np.ndarray) -> float:
    orig, pert = align_shapes(orig, pert)
    score, _ = ssim(normalize_map(orig), normalize_map(pert), data_range=1.0, full=True)
    return float(score)


def compute_spearman(orig: np.ndarray, pert: np.ndarray) -> float:
    orig, pert = align_shapes(orig, pert)
    r, _ = spearmanr(orig.flatten(), pert.flatten())
    return float(r)


def compute_pearson(orig: np.ndarray, pert: np.ndarray) -> float:
    orig, pert = align_shapes(orig, pert)
    r, _ = pearsonr(orig.flatten(), pert.flatten())
    return float(r)


def compute_centroid_displacement(orig: np.ndarray, pert: np.ndarray) -> float:
    orig, pert = align_shapes(orig, pert)
    orig_w, pert_w = np.abs(orig), np.abs(pert)
    if orig_w.sum() < 1e-12 or pert_w.sum() < 1e-12:
        return np.nan
    cy_o, cx_o = center_of_mass(orig_w)
    cy_p, cx_p = center_of_mass(pert_w)
    return float(np.sqrt((cy_o - cy_p) ** 2 + (cx_o - cx_p) ** 2))


def compute_ecd(ssim_score: float, conf_orig: float, conf_pert: float) -> float:
    return float((1.0 - ssim_score) * (conf_orig - conf_pert))


In [ ]:
df_clean = pd.read_csv(baseline_csv)
df_pert = pd.read_csv(perturbed_csv)

# Fix: clean confidence_score is a tensor repr string
df_clean['confidence_score'] = df_clean['confidence_score'].apply(parse_tensor_confidence)

# Fix: clean ground_truth_class is a broken constant string -> recover from perturbed CSV
true_labels = df_pert.drop_duplicates('image_name')[['image_name', 'ground_truth_class']]
df_clean = df_clean.drop(columns=['ground_truth_class']).merge(
    true_labels, on='image_name', how='left', validate='one_to_one'
)
print('clean rows with unrecovered label:', df_clean['ground_truth_class'].isna().sum())

# Tensor paths
df_clean['image_stem'] = df_clean['image_name'].apply(lambda n: Path(n).stem)
df_clean['tensor_path_orig'] = df_clean.apply(
    lambda r: str(CLEAN_TENSORS_BASE_DIR / r['ground_truth_class'] / f"tensor_{r['image_stem']}.npy"),
    axis=1,
)

df_pert['image_stem'] = df_pert['image_name'].apply(lambda n: Path(n).stem)
df_pert['tensor_path_pert'] = df_pert.apply(
    lambda r: str(PERTURBED_TENSORS_BASE_DIR / r['noise_type'] / r['intensity_level'] / 'test' / r['ground_truth_class'] / f"tensor_{r['image_stem']}.npy"),
    axis=1,
)

intensity_parsed = df_pert['intensity_level'].apply(parse_intensity_level).apply(pd.Series)
df_pert = pd.concat([df_pert, intensity_parsed], axis=1)

df_clean_ref = df_clean[['image_name', 'confidence_score', 'predicted_class', 'tensor_path_orig']].rename(
    columns={'confidence_score': 'conf_orig', 'predicted_class': 'predicted_class_orig'}
)
df_pert = df_pert.rename(
    columns={'confidence_score': 'conf_pert', 'predicted_class': 'predicted_class_pert', 'heatmap_path': 'heatmap_path_pert'}
)

merged = df_pert.merge(df_clean_ref, on='image_name', how='left', validate='many_to_one')
print('unmatched (no clean label found):', merged['tensor_path_orig'].isna().sum())
print('merged shape:', merged.shape)
merged.head()


clean rows with unrecovered label: 0
unmatched (no clean label found): 0
merged shape: (3000, 15)


,image_name,noise_type,intensity_level,ground_truth_class,predicted_class_pert,conf_pert,heatmap_path_pert,image_stem,tensor_path_pert,level_number,intensity_value,intensity_unit,conf_orig,predicted_class_orig,tensor_path_orig
0,998005.jpg,red,lvl_4_10dB,pizza,pizza,0.899690,output/red/lvl_4_10dB/test/pizza/heatmap_99800...,998005,/content/drive/MyDrive/study/Project/LRP_Heatm...,4,10.0,dB,0.5009,pizza,/content/drive/MyDrive/study/Project/LRP_Heatm...
1,476421.jpg,red,lvl_4_10dB,pizza,pizza,0.825237,output/red/lvl_4_10dB/test/pizza/heatmap_47642...,476421,/content/drive/MyDrive/study/Project/LRP_Heatm...,4,10.0,dB,0.8817,pizza,/content/drive/MyDrive/study/Project/LRP_Heatm...
2,2997525.jpg,red,lvl_4_10dB,pizza,pizza,0.977444,output/red/lvl_4_10dB/test/pizza/heatmap_29975...,2997525,/content/drive/MyDrive/study/Project/LRP_Heatm...,4,10.0,dB,0.9963,pizza,/content/drive/MyDrive/study/Project/LRP_Heatm...
3,3729167.jpg,red,lvl_4_10dB,pizza,pizza,0.690212,output/red/lvl_4_10dB/test/pizza/heatmap_37291...,3729167,/content/drive/MyDrive/study/Project/LRP_Heatm...,4,10.0,dB,0.9964,pizza,/content/drive/MyDrive/study/Project/LRP_Heatm...
4,648055.jpg,red,lvl_4_10dB,pizza,pizza,0.877613,output/red/lvl_4_10dB/test/pizza/heatmap_64805...,648055,/content/drive/MyDrive/study/Project/LRP_Heatm...,4,10.0,dB,0.8155,pizza,/content/drive/MyDrive/study/Project/LRP_Heatm...


In [ ]:
sample = merged.sample(min(20, len(merged)), random_state=0)
missing = 0
for _, row in sample.iterrows():
    if not Path(row['tensor_path_orig']).exists():
        print('MISSING (orig):', row['tensor_path_orig'])
        missing += 1
    if not Path(row['tensor_path_pert']).exists():
        print('MISSING (pert):', row['tensor_path_pert'])
        missing += 1
print(f'{missing} missing out of {len(sample) * 2} checked')
print('If this is 0, you are clear to run the main loop.')


0 missing out of 40 checked
If this is 0, you are clear to run the main loop.


In [ ]:
orig_tensor_cache = {}

def load_orig_cached(path_str):
    if path_str not in orig_tensor_cache:
        orig_tensor_cache[path_str] = np.load(path_str)
    return orig_tensor_cache[path_str]


records = []
missing_tensor_files = 0

for i, row in enumerate(tqdm(merged.itertuples(index=False), total=len(merged), desc='Phase 4 metrics')):
    row = row._asdict()
    if pd.isna(row['tensor_path_orig']):
        continue
    try:
        orig_map = load_orig_cached(row['tensor_path_orig'])
        pert_map = np.load(row['tensor_path_pert'])
    except FileNotFoundError:
        missing_tensor_files += 1
        continue

    try:
        ssim_score = compute_ssim(orig_map, pert_map)
        spearman_r = compute_spearman(orig_map, pert_map)
        pearson_r = compute_pearson(orig_map, pert_map)
        centroid_disp = compute_centroid_displacement(orig_map, pert_map)
        ecd = compute_ecd(ssim_score, row['conf_orig'], row['conf_pert'])

        records.append({
            'image_name': row['image_name'],
            'noise_type': row['noise_type'],
            'intensity_level': row['intensity_level'],
            'level_number': row['level_number'],
            'intensity_value': row['intensity_value'],
            'intensity_unit': row['intensity_unit'],
            'ground_truth_class': row['ground_truth_class'],
            'predicted_class_orig': row['predicted_class_orig'],
            'predicted_class_pert': row['predicted_class_pert'],
            'label_flipped': row['predicted_class_orig'] != row['predicted_class_pert'],
            'conf_orig': row['conf_orig'],
            'conf_pert': row['conf_pert'],
            'ssim': ssim_score,
            'pearson_r': pearson_r,
            'spearman_r': spearman_r,
            'centroid_displacement': centroid_disp,
            'ecd': ecd,
        })
    except Exception as e:
        print(f"Error on {row['image_name']} ({row['noise_type']}/{row['intensity_level']}): {e}")

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(records).to_csv(checkpoint_csv, index=False)

print(f'Done. {len(records)} rows computed, {missing_tensor_files} skipped (file not found).')


Phase 4 metrics:   0%|          | 0/3000 [00:00<?, ?it/s]

Done. 3000 rows computed, 0 skipped (file not found).


In [ ]:
df_results = pd.DataFrame(records)
df_results.to_csv(results_csv, index=False)
print(f'Saved {len(df_results)} rows to {results_csv}')

if len(df_results) > 0:
    summary = df_results.groupby(['noise_type', 'intensity_level'])[
        ['ssim', 'pearson_r', 'spearman_r', 'centroid_displacement', 'ecd']
    ].agg(['mean', 'std'])
    summary


Saved 3000 rows to /content/drive/MyDrive/study/Project/LRP_Heatmap/phase4_output/phase4_results.csv
